# F2 — Stage 1: head-only distillation, the gate that decides the rest

**The question this settles.** B2-bis trained an MLP (512-1024-768) on
3,000 pairs and gained **+0.003** cosine over the linear adapter, which the
report reads as proof that the relationship is linear and the residual gap
is *missing information* from MobileCLIP's 1024-to-512 bottleneck.

There is an untested alternative: the MLP may simply have been
**data-limited**. It carries ~1.31M parameters and saw 3,000 examples.
Unlike a linear map — which decomposes into independent per-output
regressions sharing one inverse, so rows per *input* dimension govern it —
an MLP's hidden layer mixes everything and does not decompose. At 3,000
examples it may have lacked the data to find any nonlinearity that exists.
This notebook repeats it with 20-50x the data, using teacher targets,
which need no annotation at all.

**Pre-registered gate** (fixed before running), measured against the
adapter's held-out numbers:

| Outcome | Reading | Next |
|---|---|---|
| cosine gain > 0.02 and R@1 gain > 0.02 | the MLP control was data-limited; the ceiling moves | proceed to F3 |
| gain <= 0.01 | the bottleneck diagnosis is confirmed by a second, stronger test | F3 justified only for the bottleneck itself |
| in between | report both, proceed cautiously | - |

Either outcome is a result. The second is arguably the more valuable: it
would mean a 0.79 MB matrix is within noise of anything a trained head can
do on the same frozen features.

**Loss.** Cosine to the teacher, plus a relational term matching the
student's in-batch similarity matrix to the teacher's - because this
project's own finding is that what matters is relational structure, not
individual coordinates.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
DATA_DIR = Path(os.environ["DATA_DIR"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"

d = np.load(str(DATA_DIR / "distill_corpus.npz"))
SIG, MOB = d["sig"], d["mob"]
tr, ev = d["train_idx"], d["eval_idx"]

# Guard: pairs.npz (used for the B3-protocol retrieval below) is
# unit-normalized at extraction. If the corpus is not, the head trains on
# one input distribution and is evaluated on another - cosine still looks
# fine, retrieval collapses to chance. Normalize here so both match.
def _unit(V):
    return (V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
            ).astype(np.float32)
_nm = np.linalg.norm(MOB, axis=1).mean()
if abs(_nm - 1.0) > 1e-3:
    print(f"corpus was raw-scale (mean norm {_nm:.3f}) - normalizing "
          "to match pairs.npz")
    SIG, MOB = _unit(SIG), _unit(MOB)
else:
    print("corpus already unit-normalized")
print(f"{len(tr)} train / {len(ev)} eval | mob {MOB.shape[1]} -> "
      f"sig {SIG.shape[1]}")

# ---- baseline to beat: the linear adapter, refit on THIS corpus ----
X = MOB[tr].astype(np.float64); Y = SIG[tr].astype(np.float64)
W = np.linalg.solve(X.T @ X + 1e-2 * np.eye(X.shape[1]), X.T @ Y)

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def recall(S):
    o = np.argsort(-S, axis=1)
    r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

base_pred = l2n(MOB[ev].astype(np.float64) @ W)
base_cos = float((base_pred * l2n(SIG[ev].astype(np.float64))).sum(1).mean())
print(f"linear adapter on this corpus: cosine {base_cos:.4f}")

In [ ]:
LAMBDA_REL = 0.5           # weight on the relational term
EPOCHS, BS, LR = 40, 512, 1e-3

class Head(nn.Module):
    def __init__(self, din, dout, hidden=2048):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(din, hidden), nn.GELU(),
                                 nn.Linear(hidden, dout))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def relational_loss(s, t):
    """match the student's in-batch geometry to the teacher's"""
    Ss, St = s @ s.T, t @ t.T
    return F.mse_loss(Ss, St)

Xtr = torch.tensor(MOB[tr]).float()
Ytr = F.normalize(torch.tensor(SIG[tr]).float(), dim=-1)
Xev = torch.tensor(MOB[ev]).float().to(DEV)
Yev = F.normalize(torch.tensor(SIG[ev]).float(), dim=-1).to(DEV)

model = Head(MOB.shape[1], SIG.shape[1]).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
best, best_state = -1, None

for ep in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(Xtr))
    for b in range(0, len(perm), BS):
        idx = perm[b:b + BS]
        x, y = Xtr[idx].to(DEV), Ytr[idx].to(DEV)
        p = model(x)
        loss = (1 - (p * y).sum(-1)).mean() + LAMBDA_REL * relational_loss(p, y)
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        c = float((model(Xev) * Yev).sum(-1).mean())
    if c > best:
        best, best_state = c, {k: v.clone() for k, v in model.state_dict().items()}
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f"  epoch {ep:3d}  held-out cosine {c:.4f}"
              f"{'  *' if c == best else ''}")
model.load_state_dict(best_state)
print(f"\nbest held-out cosine {best:.4f}  (linear {base_cos:.4f}, "
      f"gain {best-base_cos:+.4f})")

In [ ]:
# retrieval on the SAME protocol as B3, so numbers are comparable
pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad = np.load(str(DATA_DIR / "adapter.npz"))
te3 = ad["eval_idx"]
txt = l2n(pairs["sig_txt"][te3].astype(np.float64))
sig3 = l2n(pairs["sig_img"][te3].astype(np.float64))
mob3 = pairs["mob_img"][te3].astype(np.float64)

with torch.no_grad():
    stu = model(torch.tensor(mob3).float().to(DEV)).cpu().numpy()
r_stu = recall(txt @ l2n(stu).T)
r_lin = recall(txt @ l2n(mob3 @ ad["W_ridge"].astype(np.float64)).T)
r_ceil = recall(txt @ sig3.T)

print(f"{'variant':30s} R@1     R@5     R@10")
for n, r in [("ceiling (SigLIP native)", r_ceil),
             ("linear adapter (shipped, 3k)", r_lin),
             ("distilled head (F2, 38k MLP)", r_stu)]:
    print(f"{n:30s} " + "  ".join(f"{r[k]:.3f}" for k in (1,5,10)))

g_cos = best - base_cos
g_r1 = r_stu[1] - r_lin[1]
print(f"\ngains over the SHIPPED adapter: cosine {g_cos:+.4f}   "
      f"R@1 {g_r1:+.4f}")
print()
print("READ THAT R@1 GAIN WITH CARE. The shipped adapter was fitted on")
print("3,000 pairs and is LINEAR; the distilled head sees 38,000 rows and")
print("is an MLP, so the comparison moves DATA and CAPACITY at the same")
print("time. Section C.9 / F5 measured the data effect alone - the same")
print("closed-form ridge refitted on 38k rows - at R@1 +0.0244. Whatever")
print("remains after subtracting it is the capacity effect, and on 1,000")
print("eval queries the standard error of an R@1 difference is about")
print("0.022, so a residual of a point or so is not distinguishable from")
print("zero.")
print()
print("The CONTROLLED comparison is the cosine line above: linear and MLP")
print("fitted on the SAME corpus under the SAME protocol, differing only")
print("in capacity. That is the number the linearity claim rests on.")
print("\nVERDICT:", 
      "DATA-LIMITED - the MLP control understated what a head can do; "
      "proceed to F3" if (g_cos > 0.02 and g_r1 > 0.02) else
      ("BOTTLENECK CONFIRMED - a trained head on frozen features cannot "
       "beat the matrix; F3 must target the bottleneck itself"
       if g_cos <= 0.01 else "INTERMEDIATE - report both and proceed "
       "cautiously"))
torch.save(model.state_dict(), str(DATA_DIR / "f2_head.pt"))

## Reading F2

The comparison that matters is against the **linear adapter refitted on
this same corpus**, not against the original 3,000-pair fit — otherwise
the extra data would flatter the student twice. Both are trained on the
same rows and scored on the same held-out images with B3's protocol.

If the gain is small, that is the second independent confirmation that
MobileCLIP's `Linear(1024 -> 512)` bottleneck, not the mapping, sets the
ceiling — and F3 becomes an experiment about the bottleneck rather than
about capacity.